In [ ]:
# Python code that takes input as date(yyyy/mm/dd/) and ASN from a user
# Fetch ROA records from https://ftp.ripe.net/rpki/ripencc.tal/yyyy/mm/dd/roas.csv.xz. 
# It has fields: fields: URI, ASN, IP Prefix, Max Length, Not Before, Not After
# Show prefixes with that ASN
import lzma
import csv
import requests
from io import BytesIO
from datetime import datetime

def fetch_roas(date_str: str, asn_input: str):
    try:
        # Validate and parse date
        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
        url_date = date_obj.strftime('%Y/%m/%d')
        url = f'https://ftp.ripe.net/rpki/ripencc.tal/{url_date}/roas.csv.xz'
        
        print(f"Fetching ROA data from: {url}")
        response = requests.get(url)
        response.raise_for_status()

        # Decompress .xz content
        with lzma.open(BytesIO(response.content)) as f:
            csv_reader = csv.DictReader(f.read().decode().splitlines())
            
            print(f"ROA Records for ASN {asn_input} on {date_str}:\n")
            found = False
            for row in csv_reader:
                if row['ASN'].strip().upper() == asn_input.upper():
                    print(f"{row['IP Prefix']} (Max Length: {row['Max Length']})")
                    found = True

            if not found:
                print("No records found for that ASN.")
    except Exception as e:
        print(f"Error: {e}")

# Example usage
if __name__ == "__main__":
    date_input = input("Enter date (yyyy/mm/dd): ").strip()
    asn_input = input("Enter ASN (e.g., AS12345): ").strip()
    fetch_roas(date_input, asn_input)

In [ ]:
# Python code that takes input as date(yyyy/mm/dd/), prefix and ASN from a user
# Fetch ROA records from all TALs:
# https://ftp.ripe.net/rpki/ripencc.tal/yyyy/mm/dd/roas.csv.xz. 
# It has fields: fields: URI, ASN, IP Prefix, Max Length, Not Before, Not After
# Show prefixes with that ASN
import lzma
import csv
import requests
from io import BytesIO
from datetime import datetime
import ipaddress

# List of RIRs
urls = ["ripencc", "apnic", "arin", "afrinic", "lacnic"]

def fetch_roas(date_str: str, prefix_input: str, asn_input: str):
    try:
        # Parse the input date
        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
        url_date = date_obj.strftime('%Y/%m/%d')

        # Normalize prefix input
        target_prefix = ipaddress.ip_network(prefix_input, strict=False)
        found = False

        print(f"Looking for ROA of prefix {prefix_input} for ASN {asn_input} on {date_str}:\n")

        for rir in urls:
            try:
                url = f'https://ftp.ripe.net/rpki/{rir}.tal/{url_date}/roas.csv.xz'
                print(f"Checking {rir.upper()}...")

                response = requests.get(url, timeout=10)
                response.raise_for_status()

                # Decompress .xz content
                with lzma.open(BytesIO(response.content)) as f:
                    csv_reader = csv.DictReader(f.read().decode().splitlines())

                    for row in csv_reader:
                        roa_prefix = row['IP Prefix'].strip()
                        roa_asn = row['ASN'].strip().upper()
                        max_length = int(row['Max Length'])

                        if roa_asn != asn_input.upper():
                            continue

                        roa_net = ipaddress.ip_network(roa_prefix, strict=False)

                        # Skip mismatched versions (e.g., IPv4 vs IPv6)
                        if target_prefix.version != roa_net.version:
                            continue

                        # Check if target prefix is within the ROA prefix range
                        if target_prefix.subnet_of(roa_net) and target_prefix.prefixlen <= max_length:
                            print(f"✅ Found in {rir.upper()}: {roa_prefix} (Max Length: {max_length})")
                            found = True
                            
            except requests.exceptions.RequestException as e:
                print(f"  ⚠️ Failed to fetch from {rir.upper()}: {e}")
            except Exception as e:
                print(f"  ⚠️ Error processing {rir.upper()}: {e}")

        if not found:
            print("\n❌ No valid ROA found for that prefix and ASN on the given date.")
        else:
            print("\n Completed and ROA exists. ")
        
    except Exception as e:
        print(f"\nError: {e}")

# Example usage
if __name__ == "__main__":
#     date_input = input("Enter date (yyyy/mm/dd): ").strip()
#     asn_input = input("Enter ASN (e.g., AS12345): ").strip()
#     prefix_input = input("Enter prefix (e.g., 203.0.113.0/24): ").strip()
    date_input = "2025/05/09"
    prefix_input = "212.70.48.0/24"
    asn_input = "AS19905"
    fetch_roas(date_input, prefix_input, asn_input)


In [ ]:
import lzma
import csv
import requests
from io import BytesIO
from datetime import datetime
from collections import defaultdict

def fetch_roas_and_save_conflicts(date_str: str, target_asn: str):
    tal = "apnic"   
    siblings_map = {
        "AS32787": ["AS35994",  "AS16625",  "AS36183",  "AS12222",  "AS31984",  "AS17204",  "AS26008",  
                    "AS18717",  "AS393234",  "AS393560",  "AS33047",  "AS23454",  "AS36029",  "AS18680",  
                    "AS17334", "AS22207",  "AS16702",  "AS23455",  "AS22452",  "AS30675",  "AS20189",  
                    "AS35993"], # Akamai
        "AS13335": ["AS209242", "AS395747", "AS14789", "AS394536"], # Cloudflare
        "AS19905": ["AS12008",  "AS19911",  "AS397224",  "AS399163",  "AS399156",  "AS397219",  "AS399169",  "AS399170",  
                "AS399164",  "AS397233",  "AS399167",  "AS397229",  "AS397239",  "AS397222",  "AS397235",  "AS397243",  
                "AS399161",  "AS399165",  "AS397232",  "AS397238",  "AS397215",  "AS397223",  "AS399168",  "AS397225",  
                "AS399155",  "AS397218",  "AS397221",  "AS399158",  "AS397237",  "AS399154",  "AS397213",  "AS399159",  
                "AS399153",  "AS397220",  "AS397226",  "AS399160",  "AS399157",  "AS397231",  "AS397227",  "AS397241",  
                "AS397234",  "AS399173",  "AS397240",  "AS397228",  "AS397214",  "AS399177",  "AS397230",  "AS22701", 
                "AS399171",  "AS397216",  "AS397242",  "AS399180",  "AS399162",  "AS399179",  "AS397236",  "AS399176",  
                "AS399172",  "AS399175",  "AS399151",  "AS399166",  "AS397217",  "AS399178",  "AS399152",  "AS399174"
                   ], # Vercara   

        "AS19551": [], # Imperva
        "AS198949": ["AS48851", "AS213232"] # Radware
        "AS35280": ["AS43767"], # F5
        "AS20052": [] # Arbour Network (Netscout) # No prefixes registered in RPKI TALs
        "AS10690" : []
    }
    
    siblings = siblings_map.get(target_asn, [])

    try:
        # Parse date and prepare URL
        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
        url_date = date_obj.strftime('%Y/%m/%d')
        url = f'https://ftp.ripe.net/rpki/{tal}.tal/{url_date}/roas.csv.xz'
        
        print(f"Fetching ROA data from: {url}")
        response = requests.get(url)
        response.raise_for_status()

        # Decompress and read CSV
        with lzma.open(BytesIO(response.content)) as f:
            lines = f.read().decode().splitlines()
            csv_reader = csv.DictReader(lines)
            
            prefix_to_asns = defaultdict(set)

            for row in csv_reader:
                asn = row['ASN'].strip().upper()
                prefix = row['IP Prefix'].strip()
                if asn not in siblings:
                    prefix_to_asns[prefix].add(asn)

        # Filter prefixes for the target ASN
        
        target_prefixes = set()
        for prefix, asns in prefix_to_asns.items():
            if target_asn.upper() in asns:
                target_prefixes.add(prefix)

        
        # Now find how many of those prefixes also have other ASNs
        conflict_prefixes = {prefix: asns for prefix, asns in prefix_to_asns.items() 
                             if prefix in target_prefixes and len(asns) > 1}

        # Save to CSV
        output_file = 'conflicting_roas_'+tal+'_'+target_asn+'.csv'
        with open(output_file, 'w', newline='') as csvfile:
            fieldnames = ['IP Prefix', 'ASNs']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

            for prefix, asns in conflict_prefixes.items():
                
                writer.writerow({
                    'IP Prefix': prefix,
                    'ASNs': ', '.join(sorted(asns))
                })

        print(f"\nTotal prefixes registered to {target_asn}: {len(target_prefixes)}")
        print(f"Prefixes also registered to other ASNs: {len(conflict_prefixes)}")
        print(f"Saved conflicting entries to {output_file}")
    except Exception as e:
        print(f"Error: {e}")

# Example usage
if __name__ == "__main__":
#     date_input = input("Enter date (yyyy/mm/dd): ").strip()
    date_input = "2025/06/11"
#     asn_input = input("Enter ASN (e.g., AS12345): ").strip()
    asn_input = "AS13335"
    fetch_roas_and_save_conflicts(date_input, asn_input)


In [45]:
# Call RIPE stat API to get the first time seen and the last time seen
import requests
from datetime import datetime, timedelta
import pandas as pd

df = pd.read_csv("../data/as19905_may_original.csv")
prefixes = df["Prefix"]
start_times = []
end_times = [] 

for idx, prefix in enumerate(prefixes):
    
    asn_to_match = 19905
    start_time = df[df["Prefix"] == prefix]["Date"].values[0]
    start_time = str(start_time) + "T00:00:00"
    
    # Add one day 
    dt = datetime.fromisoformat(start_time)
    next_day = dt + timedelta(days=1)

    # Convert back to string in the same format
    end_time = next_day.isoformat()


    # Construct API URL
    url = (
        "https://stat.ripe.net/data/bgp-updates/data.json"
        f"?resource={prefix}&starttime={start_time}&endtime={end_time}"
    )

    # Fetch data
    response = requests.get(url)
    data = response.json()

    # Extract relevant BGP update entries
    timestamps = []

    for record in data.get("data", {}).get("updates", []):
        rec_type = record["type"]
        if rec_type == "A":
            path = record["attrs"]["path"]
            if path[-1] == asn_to_match:
                timestamps.append(record["timestamp"])

    # Report results
    if timestamps:
        # Convert strings to datetime objects
        dt_objects = [datetime.fromisoformat(ts) for ts in timestamps]

        # Find min and max
        first_seen = min(dt_objects)
        last_seen = max(dt_objects)
        
        start_times.append(first_seen)
        end_times.append(last_seen)
        print(f"For prefix {prefix} ASN {asn_to_match} seen from {first_seen} to {last_seen} UTC")
    else:
        print(f"For prefix {prefix} ASN {asn_to_match} was not seen at the end of any path.")
        
# Add StartTime and EndTime as new columns
df["StartTime"] = start_times
df["EndTime"] = end_times

# Save back to the same CSV
df.to_csv("../data/as19905_may_original.csv", index=False)
print("Completed.")

For prefix 212.70.48.0/24 ASN 19905 seen from 2025-05-09 13:39:02 to 2025-05-09 13:56:30 UTC
For prefix 46.184.88.0/24 ASN 19905 seen from 2025-05-10 01:33:02 to 2025-05-10 06:29:26 UTC
For prefix 46.184.90.0/24 ASN 19905 seen from 2025-05-10 13:17:02 to 2025-05-10 13:39:29 UTC
For prefix 212.70.48.0/24 ASN 19905 seen from 2025-05-09 13:39:02 to 2025-05-09 13:56:30 UTC
For prefix 160.62.21.0/24 ASN 19905 seen from 2025-05-12 09:19:03 to 2025-05-12 12:25:13 UTC
For prefix 204.89.59.0/24 ASN 19905 seen from 2025-05-12 15:26:02 to 2025-05-12 15:43:51 UTC
For prefix 208.94.149.0/24 ASN 19905 seen from 2025-05-13 16:18:02 to 2025-05-13 17:05:14 UTC
For prefix 78.41.60.0/24 ASN 19905 seen from 2025-05-14 10:46:02 to 2025-05-14 22:42:19 UTC
For prefix 109.230.113.0/24 ASN 19905 seen from 2025-05-14 10:37:02 to 2025-05-14 23:18:07 UTC
For prefix 193.188.60.0/24 ASN 19905 seen from 2025-05-15 12:40:03 to 2025-05-15 13:17:24 UTC
For prefix 193.188.61.0/24 ASN 19905 seen from 2025-05-15 12:40:03 

ValueError: Length of values (47) does not match length of index (49)

In [43]:
import pandas as pd
df = pd.read_csv("../data/as19905_may_original.csv")
prefixes = df["Prefix"]
for idx, prefix in enumerate(prefixes):
    start_time = df[df["Prefix"] == prefix]["Date"].values[0]
    start_time = start_time + "T00:00:00"
    print(start_time)

2025-05-09T00:00:00
2025-05-10T00:00:00
2025-05-10T00:00:00
2025-05-09T00:00:00
2025-05-12T00:00:00
2025-05-12T00:00:00
2025-05-13T00:00:00
2025-05-14T00:00:00
2025-05-14T00:00:00
2025-05-15T00:00:00
2025-05-15T00:00:00
2025-05-15T00:00:00
2025-05-15T00:00:00
2025-05-15T00:00:00
2025-05-16T00:00:00
2025-05-10T00:00:00
2025-05-17T00:00:00
2025-05-17T00:00:00
2025-05-18T00:00:00
2025-05-18T00:00:00
2025-05-20T00:00:00
2025-05-20T00:00:00
2025-05-20T00:00:00
2025-05-21T00:00:00
2025-05-21T00:00:00
2025-05-21T00:00:00
2025-05-21T00:00:00
2025-05-21T00:00:00
2025-05-21T00:00:00
2025-05-21T00:00:00
2025-05-22T00:00:00
2025-05-22T00:00:00
2025-05-23T00:00:00
2025-05-24T00:00:00
2025-05-10T00:00:00
2025-05-21T00:00:00
2025-05-26T00:00:00
2025-05-21T00:00:00
2025-05-26T00:00:00
2025-05-10T00:00:00
2025-05-12T00:00:00
2025-05-23T00:00:00
2025-05-27T00:00:00
2025-05-21T00:00:00
2025-05-18T00:00:00
2025-05-18T00:00:00
2025-05-10T00:00:00
2025-05-21T00:00:00
2025-05-30T00:00:00
